# kin_02 — Tongue latency analysis

How does tongue movement latency vary across cue-response movement ordinal (k=1, 2, 3, 4)?

**Pipeline:**
1. Load `all_tongue_movements` parquet
2. Filter to cue-response movements with valid latency
3. Latency distributions by ordinal k
4. Log-normality test for k=1 (first cue-response movement)
5. Session-level latency summaries
6. Cross-ordinal latency correlation: does k=1 latency predict k=2?

## 1. Setup

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from plotstyle import apply_style, PALETTE, style_ax, save_fig
apply_style()

In [ ]:
if Path("/root/capsule").exists():
    ENV       = "codeocean"
    SCRATCH   = Path("/root/capsule/scratch")
    FOR_LOCAL = SCRATCH / "for_local"
else:
    ENV       = "local"
    FOR_LOCAL = Path("/Users/mib/Documents/Code/kinematics_analysis/data/for_local")
    SCRATCH   = FOR_LOCAL.parent

FIG_DIR  = SCRATCH / "figures" / "kin_02_latency"
SAVE_FIG = False
print(f"ENV={ENV}")

## 2. Load data

In [ ]:
if ENV == "codeocean":
    movements_path = SCRATCH / "all_tongue_movements_04022026" / "all_tongue_movements_04022026.parquet"
else:
    movements_path = FOR_LOCAL / "all_tongue_movements_04022026.parquet"

all_tongue_movements = pd.read_parquet(movements_path)
print("Shape:", all_tongue_movements.shape)
print("Sessions:", all_tongue_movements["session"].nunique())

## 3. Filter to cue-response movements

In [ ]:
from scipy import stats as scipy_stats

# Filter to cue-response movements with valid ordinal and latency
df = (
    all_tongue_movements
    .dropna(subset=["cue_response_movement_number", "movement_latency_from_go"])
    .query("movement_latency_from_go > 0.02 and movement_latency_from_go < 2.0")
    .copy()
)
df["k"] = df["cue_response_movement_number"].astype(int)
print(df["k"].value_counts().sort_index())

## 4. Latency distributions by ordinal

In [ ]:
# Latency distributions by cue-response movement ordinal (k=1..4)
MAX_K = 4
fig, axes = plt.subplots(1, MAX_K, figsize=(4 * MAX_K, 4), sharey=False)
bins = np.arange(0, 1.55, 0.05)

for k, ax in zip(range(1, MAX_K + 1), axes):
    sub = df[df["k"] == k]["movement_latency_from_go"].to_numpy()
    sub = sub[np.isfinite(sub)]
    color = PALETTE["pos"] if k == 1 else PALETTE["neutral"]
    ax.hist(sub, bins=bins, color=color, edgecolor="white", linewidth=0.3, density=True)
    ax.axvline(np.median(sub), color=PALETTE["neg"], lw=1.2, ls="--",
               label=f"median={np.median(sub):.3f} s")
    ax.set_xlabel("Latency from go cue (s)")
    ax.set_title(f"k={k}  (n={len(sub):,})")
    ax.legend(fontsize=7)
    style_ax(ax)

axes[0].set_ylabel("Density")
fig.suptitle("Cue-response movement latency by ordinal")
plt.tight_layout()
save_fig(fig, "latency_by_ordinal", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

## 5. Log-normality test for k=1 RT

RT distributions for tongue movement onset are expected to be approximately log-normal.
This cell fits a log-normal distribution and computes normality tests on raw and log RT.

In [ ]:
# Log-normality test for k=1 latency
lat_k1 = df[df["k"] == 1]["movement_latency_from_go"].dropna().to_numpy()
lat_k1 = lat_k1[(lat_k1 > 0.02) & (lat_k1 < 2.0)]

log_lat = np.log(lat_k1)
stat_n,  p_n  = scipy_stats.normaltest(lat_k1)
stat_ln, p_ln = scipy_stats.normaltest(log_lat)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# histogram + log-normal fit
ax = axes[0]
mu_log, sig_log = np.mean(log_lat), np.std(log_lat, ddof=1)
x_range = np.linspace(0.02, 2.0, 300)
pdf_fit = scipy_stats.lognorm(s=sig_log, scale=np.exp(mu_log)).pdf(x_range)
ax.hist(lat_k1, bins=60, density=True, color=PALETTE["neutral"],
        edgecolor="white", linewidth=0.3, label="data")
ax.plot(x_range, pdf_fit, color=PALETTE["neg"], lw=1.5, label="log-normal fit")
ax.set_xlabel("Latency (s)")
ax.set_ylabel("Density")
ax.set_title(f"k=1 latency  n={len(lat_k1):,}")
ax.legend(fontsize=8)
style_ax(ax)

# log-scale histogram
ax = axes[1]
ax.hist(log_lat, bins=50, density=True, color=PALETTE["neutral"],
        edgecolor="white", linewidth=0.3, label="log(latency)")
ax.set_xlabel("log(latency) (s)")
ax.set_ylabel("Density")
ax.set_title(f"Normality test (linear): p={p_n:.3g}\nNormality test (log): p={p_ln:.3g}")
style_ax(ax)

# QQ on log-latency
ax = axes[2]
scipy_stats.probplot(log_lat, dist="norm", plot=ax)
ax.set_title("Q-Q: log(k=1 latency) vs normal")
style_ax(ax)

plt.tight_layout()
save_fig(fig, "k1_lognormal_test", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()
print(f"Normality test (raw): stat={stat_n:.3f}, p={p_n:.4g}")
print(f"Normality test (log): stat={stat_ln:.3f}, p={p_ln:.4g}")

## 6. Session-level latency summaries

In [ ]:
# Session-level latency summary per ordinal
sess_lat = (
    df
    .groupby(["session", "k"])["movement_latency_from_go"]
    .agg(["median", "count"])
    .reset_index()
    .rename(columns={"median":"med_lat","count":"n"})
)

fig, axes = plt.subplots(1, MAX_K, figsize=(4 * MAX_K, 4), sharey=True)
for k, ax in zip(range(1, MAX_K + 1), axes):
    sub = sess_lat[sess_lat["k"] == k]["med_lat"].dropna().to_numpy()
    ax.hist(sub, bins=20, color=PALETTE["neutral"], edgecolor="white", linewidth=0.3)
    ax.axvline(np.median(sub), color=PALETTE["neg"], lw=1.2, ls="--",
               label=f"median={np.median(sub):.3f} s")
    ax.set_xlabel("Session-median latency (s)")
    ax.set_title(f"k={k}  ({len(sub)} sessions)")
    ax.legend(fontsize=7)
    style_ax(ax)
axes[0].set_ylabel("Sessions")
fig.suptitle("Session-level median latency by ordinal")
plt.tight_layout()
save_fig(fig, "session_latency_by_ordinal", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

## 7. Cross-ordinal latency correlation

Does a session's median k=1 latency predict its median k=2 latency?
If yes, session-level arousal state drives latency variance across ordinals.
Note: k=1 and k=2 movements occur on *different* trials (a trial has only one
cue-response movement), so this is a session-level analysis.

In [ ]:
# Session-level: does median k=1 latency predict median k=2 latency?
# k=1 and k=2 movements are on different trials (one cue-response per trial),
# so we compare session-level medians across ordinals.
from scipy.stats import spearmanr

sess_lat = (
    df.groupby(["session", "k"])["movement_latency_from_go"]
    .median()
    .reset_index()
    .rename(columns={"movement_latency_from_go": "med_lat"})
)

sess_wide = sess_lat.pivot(index="session", columns="k", values="med_lat")

# Pairwise session-level rho between k=1 and k=2,3,4
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
compare_ks = [(1, 2), (1, 3), (2, 3)]
for ax, (ka, kb) in zip(axes, compare_ks):
    if ka not in sess_wide.columns or kb not in sess_wide.columns:
        ax.set_visible(False)
        continue
    paired = sess_wide[[ka, kb]].dropna()
    if len(paired) < 5:
        ax.set_title(f"k={ka} vs k={kb}: insufficient sessions")
        continue
    rho, p = spearmanr(paired[ka], paired[kb])
    ax.scatter(paired[ka], paired[kb], s=25, alpha=0.7, color=PALETTE["neutral"])
    ax.set_xlabel(f"Median latency k={ka} (s)")
    ax.set_ylabel(f"Median latency k={kb} (s)")
    ax.set_title(f"k={ka} vs k={kb}  ρ={rho:.3f}  p={p:.3g}  n={len(paired)}")
    style_ax(ax)
fig.suptitle("Session-level cross-ordinal latency correlation")
plt.tight_layout()
save_fig(fig, "session_rho_cross_ordinal", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()